# Experiment Notebook: RL Diploma Pipeline (Colab)

Этот ноутбук запускает весь цикл экспериментов **EXP-01..EXP-07** в Colab с устойчивым логированием и сохранением артефактов.

Что делает автоматически:
- стабилизирует окружение Colab (fix для `blinker/flask` конфликтов)
- клонирует/обновляет `develop` ветку репозитория
- запускает эксперименты по строгому порядку
- пишет логи в JSONL + сводную таблицу
- сохраняет графики, метрики и run-артефакты

> Рекомендуется запускать последовательно сверху вниз. Ячейки сделаны идемпотентными.

In [ ]:
# ===== 0) Базовая конфигурация =====
from __future__ import annotations

import os
import json
import time
import shlex
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt

CONFIG = {
    "repo_url": "https://github.com/Pancake2021/research_work_by_a_student.git",
    "repo_branch": "develop",
    "repo_dir": "/content/research_work_by_a_student",

    # Если хочешь сохранять всё в Google Drive, включи True и размонтируй/смонтируй Drive ниже.
    "use_google_drive": True,
    "drive_root": "/content/drive/MyDrive/diploma_rl",
    "local_root": "/content/diploma_rl",

    "dataset": "iemocap",           # iemocap | cmu_mosi | synthetic | local_json
    "train_size": 500,
    "test_size": 100,

    "run_all": True,
    "selected_exp_ids": ["EXP-01"],  # используется только если run_all=False

    # retries for install/network hiccups
    "max_retries": 3,
    "retry_sleep_sec": 8,
}

EXPERIMENTS = [
    {"id": "EXP-01", "mode": "baseline",    "reward": None},
    {"id": "EXP-02", "mode": "grpo",        "reward": "accuracy"},
    {"id": "EXP-03", "mode": "grpo",        "reward": "reasoning"},
    {"id": "EXP-04", "mode": "grpo",        "reward": "binary"},
    {"id": "EXP-05", "mode": "ppo",         "reward": "reasoning"},
    {"id": "EXP-06", "mode": "dapo",        "reward": "entropy"},
    {"id": "EXP-07", "mode": "lambda_grpo", "reward": "lambda_grpo"},
]


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

In [ ]:
# ===== 1) (Опционально) подключение Google Drive =====
if CONFIG["use_google_drive"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f"Drive mount skipped: {e}")

ROOT = Path(CONFIG["drive_root"] if CONFIG["use_google_drive"] else CONFIG["local_root"])
ROOT.mkdir(parents=True, exist_ok=True)

RUNS_DIR = ROOT / "runs"
RESULTS_DIR = ROOT / "results"
PLOTS_DIR = ROOT / "plots"
EXPORTS_DIR = ROOT / "exports"

for d in [RUNS_DIR, RESULTS_DIR, PLOTS_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXPERIMENT_DB = ROOT / "experiment_db.json"
if not EXPERIMENT_DB.exists():
    EXPERIMENT_DB.write_text(json.dumps({"runs": []}, ensure_ascii=False, indent=2), encoding="utf-8")

print("ROOT:", ROOT)

In [ ]:
# ===== 2) Устойчивая установка зависимостей =====

def run_bash(cmd: str, check: bool = True, capture: bool = True):
    print("$", cmd)
    p = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=capture)
    if capture and p.stdout:
        print(p.stdout[:3000])
    if p.returncode != 0:
        if capture and p.stderr:
            print(p.stderr[:4000])
        if check:
            raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p


def with_retries(fn, max_retries=3, sleep_sec=5):
    last = None
    for i in range(1, max_retries + 1):
        try:
            return fn()
        except Exception as e:
            last = e
            print(f"Attempt {i}/{max_retries} failed: {e}")
            if i < max_retries:
                time.sleep(sleep_sec)
    raise last


def install_deps():
    run_bash("python -m pip install -q --upgrade pip setuptools wheel")
    run_bash("python -m pip install -q --ignore-installed blinker")
    run_bash("python -m pip install -q flask requests psutil pyngrok")

with_retries(lambda: install_deps(), CONFIG["max_retries"], CONFIG["retry_sleep_sec"])
print("Dependencies installed")

In [ ]:
# ===== 3) Клонирование/обновление репозитория =====
repo_dir = Path(CONFIG["repo_dir"])
if not (repo_dir / ".git").exists():
    run_bash(f"rm -rf {shlex.quote(str(repo_dir))}")
    run_bash(f"git clone --branch {CONFIG['repo_branch']} {CONFIG['repo_url']} {shlex.quote(str(repo_dir))}")
else:
    run_bash(f"cd {shlex.quote(str(repo_dir))} && git fetch --all")

run_bash(f"cd {shlex.quote(str(repo_dir))} && git checkout {CONFIG['repo_branch']}")
run_bash(f"cd {shlex.quote(str(repo_dir))} && git pull --ff-only origin {CONFIG['repo_branch']}")
run_bash(f"cd {shlex.quote(str(repo_dir))} && python -m pip install -q -r requirements.txt")

print("Repo ready:", repo_dir)

In [ ]:
# ===== 4) Runtime info + DB utils =====

def get_runtime_info():
    info = {
        "status": "ok",
        "accelerator": "cpu",
        "gpu_name": None,
        "gpu_ram_gb": None,
        "ram_gb": None,
        "disk_gb": None,
    }
    try:
        import torch
        if torch.cuda.is_available():
            info["accelerator"] = "gpu"
            info["gpu_name"] = torch.cuda.get_device_name(0)
            info["gpu_ram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    except Exception:
        pass

    if os.getenv("COLAB_TPU_ADDR"):
        info["accelerator"] = "tpu"

    try:
        import psutil
        info["ram_gb"] = round(psutil.virtual_memory().total / (1024**3), 2)
        info["disk_gb"] = round(psutil.disk_usage('/').total / (1024**3), 2)
    except Exception:
        pass

    return info


def load_db():
    return json.loads(EXPERIMENT_DB.read_text(encoding="utf-8"))


def save_db(payload):
    EXPERIMENT_DB.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def append_run(run):
    db = load_db()
    db.setdefault("runs", []).append(run)
    save_db(db)


def update_run(run_id, **fields):
    db = load_db()
    for r in db.get("runs", []):
        if r.get("run_id") == run_id:
            r.update(fields)
            break
    save_db(db)

print(get_runtime_info())

In [ ]:
# ===== 5) Исполнение команды + парсинг JSON-событий =====

def run_pipeline_command(cmd: str, stdout_log: Path, stderr_log: Path):
    stdout_log.parent.mkdir(parents=True, exist_ok=True)
    stderr_log.parent.mkdir(parents=True, exist_ok=True)

    events = []
    with stdout_log.open("w", encoding="utf-8") as out_f, stderr_log.open("w", encoding="utf-8") as err_f:
        proc = subprocess.Popen(["bash", "-lc", cmd], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)

        while True:
            out_line = proc.stdout.readline() if proc.stdout else ""
            err_line = proc.stderr.readline() if proc.stderr else ""

            if out_line:
                out_f.write(out_line)
                out_f.flush()
                s = out_line.strip()
                if s.startswith("{") and s.endswith("}"):
                    try:
                        obj = json.loads(s)
                        if isinstance(obj, dict) and "event" in obj:
                            events.append(obj)
                    except Exception:
                        pass

            if err_line:
                err_f.write(err_line)
                err_f.flush()

            if not out_line and not err_line and proc.poll() is not None:
                break

        return_code = proc.wait()

    return return_code, events

In [ ]:
# ===== 6) Запуск одного эксперимента =====

def run_one_experiment(spec: dict):
    run_id = f"{spec['id'].lower()}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
    run_dir = RUNS_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    runtime = get_runtime_info()
    record = {
        "run_id": run_id,
        "exp_id": spec["id"],
        "method": spec["mode"],
        "reward_fn": spec["reward"],
        "status": "running",
        "started_at": now_iso(),
        "finished_at": None,
        "duration_hours": None,
        "runtime": runtime,
        "events": [],
        "artifacts": {},
    }
    append_run(record)

    output_dir = EXPORTS_DIR / run_id
    output_dir.mkdir(parents=True, exist_ok=True)

    reward_arg = f" --reward {spec['reward']}" if spec["reward"] else ""
    cmd = (
        f"cd {shlex.quote(CONFIG['repo_dir'])} && "
        f"python scripts/run_full_pipeline.py "
        f"--mode {spec['mode']}"
        f"{reward_arg} "
        f"--dataset {CONFIG['dataset']} "
        f"--train-size {CONFIG['train_size']} "
        f"--test-size {CONFIG['test_size']} "
        f"--run-id {run_id} "
        f"--output-dir {shlex.quote(str(output_dir))} "
        f"--json-metrics"
    )

    stdout_log = run_dir / "stdout.log"
    stderr_log = run_dir / "stderr.log"

    t0 = time.time()
    return_code, events = run_pipeline_command(cmd, stdout_log, stderr_log)
    dt_h = round((time.time() - t0) / 3600.0, 4)

    status = "success" if return_code == 0 else "failed"
    update_run(
        run_id,
        status=status,
        finished_at=now_iso(),
        duration_hours=dt_h,
        events=events,
        artifacts={
            "stdout": str(stdout_log),
            "stderr": str(stderr_log),
            "output_dir": str(output_dir),
        },
    )

    # stream JSONL for convenience
    metrics_stream = run_dir / "metrics_stream.jsonl"
    with metrics_stream.open("w", encoding="utf-8") as f:
        for ev in events:
            f.write(json.dumps(ev, ensure_ascii=False) + "\n")

    print(f"{spec['id']} => {status} (return_code={return_code})")
    if status != "success":
        print("See:", stderr_log)

    return run_id, status

In [ ]:
# ===== 7) Запуск цепочки EXP-01..EXP-07 =====
if CONFIG["run_all"]:
    plan = EXPERIMENTS
else:
    wanted = set(CONFIG["selected_exp_ids"])
    plan = [e for e in EXPERIMENTS if e["id"] in wanted]

print("Plan:", [e["id"] for e in plan])

for spec in plan:
    run_id, status = run_one_experiment(spec)
    if status != "success":
        print(f"Stopped at {spec['id']} due to failure")
        break

In [ ]:
# ===== 8) Сводная таблица по запускам =====
db = load_db()
runs = db.get("runs", [])

df = pd.DataFrame(runs)
if not df.empty:
    cols = [c for c in ["run_id", "exp_id", "method", "reward_fn", "status", "duration_hours", "started_at", "finished_at"] if c in df.columns]
    display(df[cols].sort_values("started_at", ascending=False))
else:
    print("No runs yet")

summary_csv = RESULTS_DIR / "runs_summary.csv"
df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)

In [ ]:
# ===== 9) Извлечение метрик из JSON-событий =====
db = load_db()
rows = []

for run in db.get("runs", []):
    for ev in run.get("events", []):
        row = {
            "run_id": run.get("run_id"),
            "exp_id": run.get("exp_id"),
            "method": run.get("method"),
            "reward_fn": run.get("reward_fn"),
            "event": ev.get("event"),
            "step": ev.get("step"),
            "ts_unix": ev.get("ts_unix"),
            "status": ev.get("status"),
        }

        metrics = ev.get("metrics")
        if isinstance(metrics, dict):
            for k, v in metrics.items():
                if isinstance(v, (int, float, str, bool)) or v is None:
                    row[f"m_{k}"] = v

        if ev.get("event") == "baseline_complete" and isinstance(metrics, dict):
            row["f1_weighted"] = metrics.get("f1_weighted")
            row["accuracy"] = metrics.get("accuracy")

        rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_path = RESULTS_DIR / "metrics_events.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
metrics_df.tail(20)

In [ ]:
# ===== 10) Графики =====
metrics_df = pd.read_csv(RESULTS_DIR / "metrics_events.csv") if (RESULTS_DIR / "metrics_events.csv").exists() else pd.DataFrame()

if metrics_df.empty:
    print("No metrics events yet")
else:
    # Baseline/experiment-level chart by f1 when available
    f1_df = metrics_df.dropna(subset=[c for c in ["f1_weighted"] if c in metrics_df.columns], how="all")
    if not f1_df.empty and "f1_weighted" in f1_df.columns:
        latest = (
            f1_df.sort_values("ts_unix")
            .groupby(["exp_id", "run_id"], as_index=False)
            .tail(1)
            .sort_values("exp_id")
        )

        plt.figure(figsize=(10, 4))
        plt.bar(latest["exp_id"], latest["f1_weighted"].fillna(0.0))
        plt.title("F1 by Experiment")
        plt.ylabel("F1 weighted")
        plt.grid(axis="y", alpha=0.25)
        out = PLOTS_DIR / "f1_by_experiment.png"
        plt.tight_layout()
        plt.savefig(out, dpi=180)
        plt.show()
        print("Saved:", out)

    # Event count per run (debug visibility)
    cnt = metrics_df.groupby("run_id").size().reset_index(name="n_events")
    plt.figure(figsize=(10, 4))
    plt.bar(cnt["run_id"], cnt["n_events"])
    plt.title("JSON events captured per run")
    plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", alpha=0.25)
    out2 = PLOTS_DIR / "events_per_run.png"
    plt.tight_layout()
    plt.savefig(out2, dpi=180)
    plt.show()
    print("Saved:", out2)

## Ограничения Colab и как ноутбук их снижает

- **Падения/дисконнекты:** логи и промежуточные результаты пишутся в `ROOT` после каждого запуска.
- **Нестабильный pip:** установка с retry + отдельный fix `blinker`.
- **Случайные сбои сети:** clone/pull и pip завернуты в надежные шаги.
- **Ограничение сессии:** можно запускать по одному эксперименту, изменив `run_all=False` и `selected_exp_ids`.
- **TPU:** скрипты RL в этом проекте ориентированы в первую очередь на GPU; TPU может требовать доп. адаптаций.